In [1]:
import sys
import numpy as np
import pandas as pd

from typing import Literal
from collections import Counter
from ase import Atoms
from ase.neighborlist import natural_cutoffs
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.core.surface import Slab, Structure
from pymatgen.core import Composition
from pymatgen.core.surface import (
    Slab, get_symmetrically_distinct_miller_indices, SlabGenerator)
from ase.neighborlist import NeighborList, natural_cutoffs
from ase.io import read, Trajectory

ZrO2_m = read('ZrO2_m.cif')
TiO2_r = read('TiO2_rutile.cif')
test_cands = Trajectory('test_cands.traj')

atoms = test_cands[18]

S = AseAtomsAdaptor.get_structure(atoms)

num_layers = 1
while True:
    sg = SlabGenerator(
    initial_structure = S.copy(),
    miller_index = (0,1,0),
    min_slab_size = num_layers,
    min_vacuum_size = 2,
    lll_reduce = True,
    in_unit_planes = True,
    center_slab = True,
    primitive = True,
    )
    slabs = sg.get_slabs(symmetrize = True)
    if len(slabs) == 0:
        num_layers += 1
        continue


    list_of_atoms = []
    for slab in slabs:
        list_of_atoms.append(AseAtomsAdaptor.get_atoms(slab))

    thicknesses = []
    for atoms in list_of_atoms:
        z_pos = atoms.get_positions()[:,2]
        z_min = np.min(z_pos)
        z_max = np.max(z_pos)
        thicknesses.append(z_max - z_min)
        print(z_max - z_min)
    
    print('N slabs:', len(slabs), 'N atoms:', len(list_of_atoms), 'N thicknesses:', len(thicknesses))

    min = np.min(thicknesses)
    print(f"Minimum thickness: {np.round(min, 2)} Å")
    if min >= 9:
        break
    else:
        print('-------------------------')
        num_layers += 1
        print(num_layers)
        print('-------------------------')
    


from ase.visualize import view
# view(list_of_atoms)

c:\Users\matnis\OneDrive - Danmarks Tekniske Universitet\Skrivebord\Projects\GNoME_DREAMS_OER_screening\.venv\Lib\site-packages\pymatgen\core\surface.py:1362: UserWarning: Too many sites removed, please use a larger slab.
  sym_slabs = self.nonstoichiometric_symmetrized_slab(group[0])
c:\Users\matnis\OneDrive - Danmarks Tekniske Universitet\Skrivebord\Projects\GNoME_DREAMS_OER_screening\.venv\Lib\site-packages\pymatgen\core\surface.py:1362: UserWarning: Too many sites removed, please use a larger slab.
  sym_slabs = self.nonstoichiometric_symmetrized_slab(group[0])
c:\Users\matnis\OneDrive - Danmarks Tekniske Universitet\Skrivebord\Projects\GNoME_DREAMS_OER_screening\.venv\Lib\site-packages\pymatgen\core\surface.py:1362: UserWarning: Too many sites removed, please use a larger slab.
  sym_slabs = self.nonstoichiometric_symmetrized_slab(group[0])


11.75884374476316
8.486848443103671
6.441705140586878
10.866335765350895
7.566271676997478
6.44170514058688
N slabs: 6 N atoms: 6 N thicknesses: 6
Minimum thickness: 6.44 Å
-------------------------
3
-------------------------
16.783167360189722
14.615157125105696
12.959281206712083
16.313370540507876
13.681860514821725
12.959281206712085
N slabs: 6 N atoms: 6 N thicknesses: 6
Minimum thickness: 12.96 Å


In [2]:
temp_dict = {}

len(temp_dict)

0

In [9]:
cand =  test_cands[0] # TiO2_r

In [10]:
from gnome_dreams_oer_screening.oer.oer_study_utils import (
    eval_stoichiometry_diff, get_stoichiometry_diff,
    get_stoichiometry_dict)
from gnome_dreams_oer_screening import OER_catalyst_study

atoms_list = []
study = OER_catalyst_study(cand,     
    H2O_gas_free_energy = -14.217,
    H2_gas_free_energy = -6.77
)

slabs = study._build_surfaces((1,0,1), stoichiometry_tolerance = 0.2,
    all_species_present = True,
    select_closest_O_stoichiometry = True,
    max_atoms=1000, max_slab_thickness=1000,
    ignore_stoichiometry_of_species = [])

for slab in slabs:
    print(study.compute_slab_thickness(slab))
    atoms_list.append(AseAtomsAdaptor.get_atoms(slab))

        
view(atoms_list)

Selected 2 slabs out of  8 based on closest O-stoichiometry.
12.874051298885838
12.772720597681072


<Popen: returncode: None args: ['c:\\Users\\matnis\\OneDrive - Danmarks Tekn...>

In [ ]:
for slab in slabs:
    print(len(slab))

48


In [ ]:
for slab in slabs:
    d = get_stoichiometry_diff(
        get_stoichiometry_dict(study.bulk_structure),
        get_stoichiometry_dict(slab)) 
    print(d['O'])
    print( eval_stoichiometry_diff(study.bulk_structure, slab,
                                max_diff_dict={'O': 0.02},   
                                )
           )
    print('_____________________________________')


0.0
True
_____________________________________
